In [846]:
print(" -------- Task 1 --------- ")
csv_messy_path = r"..\data\messy\messy_market_data.csv"
print(f"loaded: {csv_messy_path}")

 -------- Task 1 --------- 
loaded: ..\data\messy\messy_market_data.csv


In [847]:
import pandas as pd

messy_data_df = pd.read_csv(csv_messy_path)

rows, columns = messy_data_df.shape

print(f"Rows: {rows}")
print(f"Columns: {columns}")

Rows: 9991
Columns: 13


In [848]:
print("First 10 rows of messy data shown:")

first_10_rows_of_messy_data = messy_data_df.head(10)

print(first_10_rows_of_messy_data)

First 10 rows of messy data shown:
      symbol interval                  open_time            open  \
0   BTCUSDT        1h  2026-05-29 23:00:00+00:00  73483.51000000   
1    BTCUSDT       1h  2026-06-07 19:00:00+00:00  62030.15000000   
2   DOGEUSDT       1h  2026-06-23 06:00:00+00:00      0.08172000   
3   LINKUSDT       1h  2026-05-25 18:00:00+00:00      9.60500000   
4    DOTUSDT       1h  2026-05-29 04:00:00+00:00      1.20400000   
5   DOGEUSDT       1h  2026-05-21 08:00:00+00:00      0.10564000   
6   linkusdt       1h  2026-05-31 15:00:00+00:00      9.09900000   
7   LINKUSDT       1h  2026-06-25 09:00:00+00:00      7.50800000   
8   DOGEUSDT       1h  2026-05-22 10:00:00+00:00      0.10575000   
9    BNBUSDT       1h  2026-06-18 10:00:00+00:00    590.67000000   

             high             low           close             volume  \
0  73510.00000000  73320.00000000  73460.78000000       271.23268000   
1  62036.78000000  61184.00000000  61328.00000000      1046.65633000   


In [849]:
messy_data_datatypes = messy_data_df.dtypes

print(messy_data_datatypes)

symbol                    str
interval                  str
open_time                 str
open                      str
high                      str
low                       str
close                     str
volume                    str
close_time                str
quote_volume              str
trade_count               str
taker_buy_base_volume     str
taker_buy_quote_volume    str
dtype: object


In [850]:
print(" -------- Task 2 --------- ")

messy_data_df_missing_mask = messy_data_df.isnull() # by columns

messy_data_df_missing_counts = messy_data_df_missing_mask.sum() # counts by columns

messy_data_df_missing_mask_rows = messy_data_df_missing_mask.any(axis=1) # by rows

messy_data_df_missing_mask_row_counts = messy_data_df_missing_mask_rows.sum() # row counts

sorted_messy_data_missing_counts = messy_data_df_missing_counts.sort_values(ascending=False)

print("All missing values sorted")

print(sorted_messy_data_missing_counts)


 -------- Task 2 --------- 
All missing values sorted
close_time                58
high                      57
quote_volume              51
open_time                 47
trade_count               44
taker_buy_base_volume     43
low                       41
volume                    41
taker_buy_quote_volume    40
open                      38
close                     36
interval                   0
symbol                     0
dtype: int64


In [851]:
print("Most missing data are in the below 3 columns with missing value counts: ")

messy_missing_counts_top_3 = sorted_messy_data_missing_counts.head(3)

print(messy_missing_counts_top_3)

# To get most impacted column: 

messy_missing_counts_top_1 = sorted_messy_data_missing_counts.head(1)

print(f"Most missing data are for most impacted column name: {messy_missing_counts_top_1.index[0]}")

Most missing data are in the below 3 columns with missing value counts: 
close_time      58
high            57
quote_volume    51
dtype: int64
Most missing data are for most impacted column name: close_time


In [852]:
# ---- Start of cleaning process ----

for_cleaning_data_df = messy_data_df.copy()

columns_to_convert_to_numeric = ["open", "close", "high", "low", "trade_count", "volume", "quote_volume", "taker_buy_base_volume", "taker_buy_quote_volume"]

# Get missing ones as mask which keeps marks ones were missing

cleaning_messy_originally_missing_df = for_cleaning_data_df[columns_to_convert_to_numeric].isnull() # to find missing values before coerce changes conversion invalid also as NaN


# Convert and change invalid to NaN also

cleaning_messy_converted_to_numeric = for_cleaning_data_df[columns_to_convert_to_numeric].apply(pd.to_numeric, errors = 'coerce')


# Get inclusive mask with missing and invalid 
Marked_missing_plus_conversion_invalid_as_missing_df = cleaning_messy_converted_to_numeric[columns_to_convert_to_numeric].isnull() # to find as missing - NaN (marked Nan from coerce) also ones from invalid conversion


# Get mask with invalid ones and marks which ones were invalid by subtracting missing mask from total mask

messy_data_only_conversion_numeric_invalid = Marked_missing_plus_conversion_invalid_as_missing_df & ~ cleaning_messy_originally_missing_df

messy_data_only_conversion_numeric_invalid_mask = messy_data_only_conversion_numeric_invalid.any(axis=1)

messy_data_only_conversion_numeric_invalid_row_count = messy_data_only_conversion_numeric_invalid_mask.sum()

#

for_cleaning_data_df[columns_to_convert_to_numeric] = cleaning_messy_converted_to_numeric

print("""
Below post conversion datatypes of columns.
The invalid and missing are saved in dataframe masks for reference.
""")

print(for_cleaning_data_df.dtypes)

print(f"""
Attributes converted are: {", ".join(columns_to_convert_to_numeric)}.
""")




Below post conversion datatypes of columns.
The invalid and missing are saved in dataframe masks for reference.

symbol                        str
interval                      str
open_time                     str
open                      float64
high                      float64
low                       float64
close                     float64
volume                    float64
close_time                    str
quote_volume              float64
trade_count               float64
taker_buy_base_volume     float64
taker_buy_quote_volume    float64
dtype: object

Attributes converted are: open, close, high, low, trade_count, volume, quote_volume, taker_buy_base_volume, taker_buy_quote_volume.



In [853]:
# Now have to count rows which have at least one NaN or true for isnull() as opposed to cells.
# Marked_missing_plus_conversion_invalid_as_missing_df This True False Dataframe has True for missing or conversion_invalid. We group them all as invalid/missing

# print(Marked_missing_plus_conversion_invalid_as_missing_df.sum()) # We sum the invalid/missing -- But this would be by column we want by row

check_marked_missing_plus_invalid_in_row = Marked_missing_plus_conversion_invalid_as_missing_df.any(axis=1) # We check if invalid/missing (True) on the row - req is by row and not by column hence we focus on dataframe axis

missing_invalid_row_count = check_marked_missing_plus_invalid_in_row.sum()

print(f"Numeric Invalid/Missing numeric rows after conversion: {missing_invalid_row_count}.")

Numeric Invalid/Missing numeric rows after conversion: 770.


In [854]:
columns_convert_to_timestamp = ["open_time", "close_time"]

print("""
Task 4: Converting time attributes to timestamp. 
Then marking invalid to conversion values for capturing.
""")

cleaning_messy_converted_to_timestamp_df = for_cleaning_data_df[columns_convert_to_timestamp].apply(pd.to_datetime, errors = 'coerce')

print(f"Converted to timestamp columns: {", ".join(columns_convert_to_timestamp)}")

# To count the invalid values which are Nan after the coerse I do isnull()

cleaning_messy_converted_to_timestamp_mark_invalid_ones_as_True_mask = cleaning_messy_converted_to_timestamp_df.isnull() # mark invalid from conversion to timestamp by columns

cleaning_messy_converted_to_timestamp_invalid_rows = cleaning_messy_converted_to_timestamp_mark_invalid_ones_as_True_mask.any(axis=1)  # mark invalid from conversion to timestamp rows

cleaning_messy_converted_to_timestamp_mark_invalid_ones_as_True_row_count = cleaning_messy_converted_to_timestamp_invalid_rows.sum() # invalid from conversion to timestamp count rows

print("After conversion, below are invalid to timestamp conersion counts:")

open_time_invalid_counts, close_time_invalid_counts = cleaning_messy_converted_to_timestamp_mark_invalid_ones_as_True_mask.sum()

print(f"Invalid open_time counts: {open_time_invalid_counts}.")
print(f"Invalid close_time counts: {close_time_invalid_counts}.\n")

for_cleaning_data_df[columns_convert_to_timestamp] = cleaning_messy_converted_to_timestamp_df # Put the applied with conversion df into the cleaned df.

print("Validating after new conversion cleaned df datatypes:\n")
print(for_cleaning_data_df.dtypes)



Task 4: Converting time attributes to timestamp. 
Then marking invalid to conversion values for capturing.

Converted to timestamp columns: open_time, close_time
After conversion, below are invalid to timestamp conersion counts:
Invalid open_time counts: 199.
Invalid close_time counts: 205.

Validating after new conversion cleaned df datatypes:

symbol                                    str
interval                                  str
open_time                 datetime64[us, UTC]
open                                  float64
high                                  float64
low                                   float64
close                                 float64
volume                                float64
close_time                datetime64[us, UTC]
quote_volume                          float64
trade_count                           float64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
dtype: object


In [855]:
# Now cleaning and standardising the format of values of symbol attribute:

print("Now cleaning and standardising the format of values of symbol attribute.\n")

symbol_columns_for_cleaning_and_standardization = ["symbol"]

# Printing symbol values before cleaning:

print("Below are symbol value counts before cleaning:")


print(for_cleaning_data_df["symbol"].value_counts())


unique_symbol_values_before_cleaning = for_cleaning_data_df["symbol"].sort_values().unique() # Symbol values before cleaning.

print(f"\nSymbol values before cleaning: {", ".join(unique_symbol_values_before_cleaning)}")

# Below are the cleaning processing steps for symbol:

cleaning_and_standardizing_symbol_values_df = for_cleaning_data_df["symbol"].str.strip().str.upper().str.replace("/","")

for_cleaning_data_df["symbol"] = cleaning_and_standardizing_symbol_values_df

# Printing

print("\nBelow are symbol value counts after cleaning:")

print(f"{for_cleaning_data_df["symbol"].value_counts()}\n")

unique_symbol_values_after_cleaning = for_cleaning_data_df["symbol"].sort_values().unique()  # Symbol values after cleaning.

print(f"Symbol values after cleaning: {', '.join(unique_symbol_values_after_cleaning)}\n")

print(f"Unique symbols count after cleaning: {len(unique_symbol_values_after_cleaning)}")

Now cleaning and standardising the format of values of symbol attribute.

Below are symbol value counts before cleaning:
symbol
AVAXUSDT      978
XRPUSDT       970
DOTUSDT       965
ETHUSDT       964
ADAUSDT       963
DOGEUSDT      959
BNBUSDT       955
LINKUSDT      952
BTCUSDT       950
SOLUSDT       936
BTC/USDT       30
SOL/USDT       21
ADA/USDT       19
linkusdt       18
ETH/USDT       18
DOGE/USDT      17
 AVAXUSDT      17
 ADAUSDT       15
 ETHUSDT       15
 SOLUSDT       14
avaxusdt       14
ethusdt        14
LINK/USDT      14
adausdt        14
DOT/USDT       13
 DOGEUSDT      12
XRP/USDT       12
BNB/USDT       11
AVAX/USDT      11
dotusdt        10
btcusdt        10
dogeusdt       10
 LINKUSDT      10
 BNBUSDT       10
solusdt         9
 XRPUSDT        9
 DOTUSDT        9
 BTCUSDT        8
xrpusdt         8
bnbusdt         7
Name: count, dtype: int64

Symbol values before cleaning:  ADAUSDT ,  AVAXUSDT ,  BNBUSDT ,  BTCUSDT ,  DOGEUSDT ,  DOTUSDT ,  ETHUSDT ,  LINKUSDT ,  SO

In [856]:
# Task 5 for Counting and removing duplicate rows.

print ("""
---- Task 5 ---
Counting and removing duplicates.""")

duplicates_mask = for_cleaning_data_df.duplicated()

duplicates_row_count = duplicates_mask.sum()

print(f"Duplicate rows found: {duplicates_row_count}.")

#Row counts before duplicates removed.

messy_rows_counts_before_duplicates_removal = len(for_cleaning_data_df)

rows_after_duplicates_removal = for_cleaning_data_df.drop_duplicates()

rows_counts_after_duplicates_removal = len(rows_after_duplicates_removal)

for_cleaning_data_df = rows_after_duplicates_removal

print(f"Rows before removing duplicates: {messy_rows_counts_before_duplicates_removal}.")

print(f"Rows after removing duplicates: {rows_counts_after_duplicates_removal}.")

# print(len(for_cleaning_data_df)) # For checking the values of for_cleaning_data_df after duplicate rows removal



---- Task 5 ---
Counting and removing duplicates.
Duplicate rows found: 213.
Rows before removing duplicates: 9991.
Rows after removing duplicates: 9778.


In [857]:
# Task 6 -- Impossible Values checks

# 3 layer-step approach is followed. 
# One is to create a mask as variable with filter-condition, either with axis=1 when multiple columns are to create a row_based pandas series mask or otherwise directly if already series.
# Second is to use mask to find rows if required for future reference or for checking.
# Third is to use the mask to count where the condition is true.

# Detecting negative volume rows:

columns_negative_check_volume = ["volume", "quote_volume", "taker_buy_base_volume", "taker_buy_quote_volume"]

detecting_negative_volume_rows = for_cleaning_data_df[columns_negative_check_volume]<0

detecting_negative_volume_rows_mask = detecting_negative_volume_rows.any(axis=1)

negative_volume_row_count = detecting_negative_volume_rows_mask.sum()

print(f"Negative volume rows: {negative_volume_row_count} .")


# Detecting negative trade_count rows:

columns_negative_check_count = ["trade_count"]

detecting_negative_trade_count_rows = for_cleaning_data_df[columns_negative_check_count]<0

detecting_negative_trade_count_rows_mask = detecting_negative_trade_count_rows.any(axis=1)

negative_trade_row_count = detecting_negative_trade_count_rows_mask.sum()

print(f"Negative trade_count rows: {negative_trade_row_count} .")


# Detecting high lower than low price rows:

high_minus_low_difference = for_cleaning_data_df["high"] - for_cleaning_data_df["low"]

high_minus_low_difference_mask = high_minus_low_difference<0

high_minus_low_difference_rows = for_cleaning_data_df[high_minus_low_difference_mask] # To get the rows where high lower than low.

high_minus_low_difference_row_count = high_minus_low_difference_mask.sum() #To get the count of rows with high lower than low.

print(f"Rows where high < low: {high_minus_low_difference_row_count} .")

# Detect rows where any of 3 masks are true. If two on same row not to double count.

Negative_trade_count_or_volume_or_lower_high_mask = detecting_negative_volume_rows_mask | detecting_negative_trade_count_rows_mask | high_minus_low_difference_mask

Negative_trade_count_or_volume_or_lower_high_rows = for_cleaning_data_df[Negative_trade_count_or_volume_or_lower_high_mask]

Negative_trade_count_or_volume_or_lower_high_row_count = Negative_trade_count_or_volume_or_lower_high_mask.sum()

print(f"Invalid numeric rows total: {Negative_trade_count_or_volume_or_lower_high_row_count} .")





Negative volume rows: 299 .
Negative trade_count rows: 0 .
Rows where high < low: 0 .
Invalid numeric rows total: 299 .


In [858]:
#Task 7
import numpy as np

print("----- Task 7 -----")
print("Adding calculated columns.")

created_columns = ["price_range", "price_change", "percent_change", "candle_direction"]

for_cleaning_data_df["price_range"] = for_cleaning_data_df["high"] - for_cleaning_data_df["low"]
for_cleaning_data_df["price_change"] = for_cleaning_data_df["close"] - for_cleaning_data_df["open"]
for_cleaning_data_df["percent_change"] = (for_cleaning_data_df["price_change"] / for_cleaning_data_df["open"])*100

conditions_for_candle_direction = [for_cleaning_data_df["price_change"]>0, for_cleaning_data_df["price_change"]==0, for_cleaning_data_df["price_change"]<0]
values_for_candle_direction = ["up", "flat", "down"]

for_cleaning_data_df["candle_direction"] = np.select(
    conditions_for_candle_direction,
    values_for_candle_direction,
    default = "unknown direction"
)

print("Created columns: ", ", ".join(created_columns))

# Now will print an example row

index_example_row = 0
example_row = for_cleaning_data_df.iloc[index_example_row]

print(f"""
Example Row (index '{index_example_row}'): open={example_row["open"]} close={example_row["close"]} high={example_row["high"]} low={example_row["low"]} 
price_range={example_row["price_range"]} price_change={example_row["price_change"].round(2)} percent_change={example_row["percent_change"].round(2)}% candle_direction={example_row["candle_direction"]}
""")

----- Task 7 -----
Adding calculated columns.
Created columns:  price_range, price_change, percent_change, candle_direction

Example Row (index '0'): open=73483.51 close=73460.78 high=73510.0 low=73320.0 
price_range=190.0 price_change=-22.73 percent_change=-0.03% candle_direction=down



In [859]:
print("----- Task 8 ----- ")
print("Data Quality Report:\n")

# messy_data_df
# final_cleaned_data_df

messy_data_row_count = len(messy_data_df)

final_cleaned_data_df_row_count = len(final_cleaned_data_df)

print(f"Rows before cleaning: {messy_data_row_count}")

# print(f"Rows after cleaning: {final_cleaned_data_df_row_count}")


print(f"Rows with missing values before cleaning: {messy_data_df_missing_mask_row_counts}")

print(f"Rows with invalid numeric values before cleaning: {messy_data_only_conversion_numeric_invalid_row_count}")

print(f"Rows with invalid timestamp values before cleaning: {cleaning_messy_converted_to_timestamp_mark_invalid_ones_as_True_row_count}")

print(f"Rows with duplicates before cleaning: {duplicates_row_count}")

print(f"Rows with impossible volumes, trade_counts, and price movements before cleaning: {Negative_trade_count_or_volume_or_lower_high_row_count}")

# final_cleaned_data_df_missing_values_count




----- Task 8 ----- 
Data Quality Report:

Rows before cleaning: 9991
Rows with missing values before cleaning: 496
Rows with invalid numeric values before cleaning: 399
Rows with invalid timestamp values before cleaning: 402
Rows with duplicates before cleaning: 213
Rows with impossible volumes, trade_counts, and price movements before cleaning: 299


In [860]:
# Task 8b: Cleaning and Repairs after the duplicates removals, datatype conversions, calculated fields, and pre processing.

cleaned_data_removed_duplicates_df = for_cleaning_data_df
print(f"Rows after removing duplicates: {len(cleaned_data_removed_duplicates_df)}")

# Removal of rows with missing values
cleaned_data_removed_missing_df = cleaned_data_removed_duplicates_df[~messy_data_df_missing_mask_rows]
print(f"Rows after removing missing values: {len(cleaned_data_removed_missing_df)}")

# Removal of rows with invalid numeric values
cleaned_data_removed_invalid_numeric_df = cleaned_data_removed_missing_df[~messy_data_only_conversion_numeric_invalid_mask]
print(f"Rows after removing invalid numeric values: {len(cleaned_data_removed_invalid_numeric_df)}")

# Removal of rows with invalid timestamp values
cleaned_data_removed_invalid_timestamp_df = cleaned_data_removed_invalid_numeric_df[~cleaning_messy_converted_to_timestamp_invalid_rows]
print(f"Rows after removing invalid timestamp values: {len(cleaned_data_removed_invalid_timestamp_df)}")

# Removal of rows with impossible volumes, trade_counts, or price_movements
cleaned_data_removed_impossible_volumns_trade_counts_price_movements_df = cleaned_data_removed_invalid_timestamp_df[~Negative_trade_count_or_volume_or_lower_high_mask]
print(f"Rows after removing impossible (negative) volumes, trade counts, price_movements: {len(cleaned_data_removed_impossible_volumns_trade_counts_price_movements_df)}")

# Remaining row counts
final_cleaned_data_df = cleaned_data_removed_impossible_volumns_trade_counts_price_movements_df
print(f"Rows of final cleaned dataset: {len(final_cleaned_data_df)}")


Rows after removing duplicates: 9778
Rows after removing missing values: 9282
Rows after removing invalid numeric values: 8908
Rows after removing invalid timestamp values: 8638
Rows after removing impossible (negative) volumes, trade counts, price_movements: 8364
Rows of final cleaned dataset: 8364


C:\Users\Markos\AppData\Local\Temp\ipykernel_34928\936578063.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cleaned_data_removed_missing_df = cleaned_data_removed_duplicates_df[~messy_data_df_missing_mask_rows]
C:\Users\Markos\AppData\Local\Temp\ipykernel_34928\936578063.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cleaned_data_removed_invalid_numeric_df = cleaned_data_removed_missing_df[~messy_data_only_conversion_numeric_invalid_mask]
C:\Users\Markos\AppData\Local\Temp\ipykernel_34928\936578063.py:15: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cleaned_data_removed_invalid_timestamp_df = cleaned_data_removed_invalid_numeric_df[~cleaning_messy_converted_to_timestamp_invalid_rows]
C:\Users\Markos\AppData\Local\Temp\ipykernel_34928\936578063.py:19: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cleaned_data_removed_impossible_volumns_trade_counts_

In [861]:
# Task 8c: Counting categories after cleaning

# Total Row Counts after cleaning

final_cleaned_data_df_row_counts = len(final_cleaned_data_df)

print(f"Rows after cleaning: {final_cleaned_data_df_row_counts}")

# Count duplicates

final_cleaned_data_df_duplicates_mask = final_cleaned_data_df.duplicated()

final_cleaned_data_df_duplicates_row_counts = final_cleaned_data_df_duplicates_mask.sum()

print(f"Duplicates after cleaning: {final_cleaned_data_df_duplicates_row_counts}")

# Count missing, invalid numeric, invalid timestamps

final_cleaned_data_df_missing_or_post_conversion_invalid = final_cleaned_data_df.isnull().any(axis=1)

final_cleaned_data_df_missing_or_post_conversion_invalid_row_counts = final_cleaned_data_df_missing_or_post_conversion_invalid.sum()

print(f"Missing, invalid numerics and invalid timestamp counts after cleaning: {final_cleaned_data_df_missing_or_post_conversion_invalid_row_counts}")

# Count impossible (negative) volumes, trade_counts, price movements (price_range)

columns_negative_check = ["volume", "quote_volume", "taker_buy_base_volume", "taker_buy_quote_volume", "trade_count", "price_range"]

detecting_impossible_negative_values_df = final_cleaned_data_df[columns_negative_check]<0

detecting_impossible_negative_values_mask = detecting_impossible_negative_values_df.any(axis=1)

detecting_impossible_negative_values_rows = final_cleaned_data_df[detecting_impossible_negative_values_mask]

detecting_impossible_negative_values_row_counts = detecting_impossible_negative_values_mask.sum()

print(f"Impossible negative volumes, trade_counts, price movements (price_range) after cleaning: {detecting_impossible_negative_values_row_counts}")


Rows after cleaning: 8364
Duplicates after cleaning: 0
Missing, invalid numerics and invalid timestamp counts after cleaning: 0
Impossible negative volumes, trade_counts, price movements (price_range) after cleaning: 0


In [862]:
# Save cleaned dataframe final_cleaned_data_df

csv_final_cleaned_output_path = r"..\data\clean\cleaned_market_data.csv"

final_cleaned_data_df.to_csv(csv_final_cleaned_output_path, index=False)

print(f"Saved cleaned dataset: {csv_final_cleaned_output_path}")

Saved cleaned dataset: ..\data\clean\cleaned_market_data.csv


In [ ]:
# Sample Check:

loaded_cleaned_data_for_sample_df = pd.read_csv(csv_final_cleaned_output_path)


symbol_sample_rows_df = loaded_cleaned_data_for_sample_df.groupby("symbol").sample(n=5)

print(symbol_sample_rows_df.shape)
print(symbol_sample_rows_df)


TypeError: 'tuple' object is not callable